# Project Title: Alternative Credit Scoring System
## Subtitle: A Predictive Model for Household Loan Default Risk using the 2021 FinAccess Survey
**Author:** PAUL – Level 300 Applied Math & CS, Kenyatta University

---

### 1. Project Overview
This project develops a **practitioner-level credit scoring engine** designed to assess the creditworthiness of Kenyan individuals. By leveraging the **2021 FinAccess Household Survey**, the model moves beyond traditional collateral-based lending to explore **alternative data**—including mobile money usage (M-Shwari, Fuliza), psychographic financial behavior, and household economic resilience.

### 2. The Problem Statement
Despite high mobile money penetration in Kenya, a significant portion of the population remains **"thin-file"** (lacking a traditional credit history). For institutions like Safaricom or Tier 1 Banks, the challenge is to accurately predict the probability of default ($PD$) using survey-based behavioral proxies. Inaccurate scoring leads to high **Non-Performing Loans (NPLs)** or financial exclusion for deserving borrowers.

### 3. Objectives
* **Target Definition:** Quantify "Financial Distress" using markers like late repayments, asset liquidation, and meal-skipping due to debt.
* **Feature Engineering:** Extract high-signal predictors from mobile banking frequency, informal (*Chama*) involvement, and demographic stability.
* **Modeling:** Train a binary classification model to differentiate between "Low Risk" and "High Risk" individuals.
* **Deployment Readiness:** Evaluate the model using banking metrics: ROC-AUC, Gini Coefficient, and Precision-Recall curves.

### 4. Data Source
The dataset is the **2021 FinAccess Household Survey (Weighted)**, a national representative survey of 22,000+ households conducted by the **Kenya National Bureau of Statistics (KNBS)** and the **Central Bank of Kenya (CBK)**.

##  PHASE 1:DATA CLEANING

## 1. Data loading

In [1]:
# importing the libraries
import pandas as pd
import pyreadstat

In [2]:
# we load the data and the metadata
df,meta =pyreadstat.read_sav('C:/Users/Win/Alternative credit scoring/Data/Updated Anonymized Weighted FinAccess 2021.sav')

In [3]:
#lets preview or dataset
df.head()

,County,ClusterNo,HHNo,interview__key,interview__id,A9,A9i,A10i,A14v,A14vi,...,excl_mob_bank_acc2,excl_fuliza,excl_bank_mobile_users2,comb_mob_bank_acc,uninsured_NHIF,Saving_groups,A1,has_tele,gave_tele,hasagr
0,26.0,10226038.0,1048.0,10-67-89-46,0003fc74b3fe418ea041bd6a9e7ff387,1.0,1.0,2.0,1.0,1.0,...,0.0,0.0,1.0,0.0,0.0,1.0,TRANS NZOIA,1.0,1.0,1.0
1,40.0,10240034.0,1080.0,39-64-68-81,0004890b17744272baf0a0c7b4c20771,1.0,1.0,2.0,4.0,2.0,...,0.0,0.0,4.0,0.0,0.0,1.0,BUSIA,1.0,1.0,0.0
2,16.0,10216062.0,1013.0,92-34-74-01,00052153fe8c4abaa189caadcb87b2b4,1.0,1.0,1.0,1.0,1.0,...,0.0,0.0,4.0,0.0,0.0,1.0,MACHAKOS,0.0,0.0,1.0
3,42.0,10242078.0,1026.0,08-14-22-63,000d1f8747194b6a84862830dc5fe7ca,1.0,1.0,1.0,5.0,4.0,...,0.0,0.0,4.0,0.0,1.0,0.0,KISUMU,1.0,1.0,0.0
4,21.0,10221166.0,1096.0,03-40-25-90,0010697b1e454d1cb38c60acacc249dd,2.0,1.0,1.0,2.0,1.0,...,0.0,0.0,4.0,1.0,1.0,1.0,MURANG'A,1.0,1.0,0.0


In [4]:
#lets see the shape of our dataset
print(f" The dataset has: (rows , columns) {df.shape}")

 The dataset has: (rows , columns) (22024, 2353)


In [5]:
#lets get a simple summary of our dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22024 entries, 0 to 22023
Columns: 2353 entries, County to hasagr
dtypes: float64(2190), object(163)
memory usage: 395.4+ MB


In [6]:
#lets check for duplicates
df.duplicated().sum()

np.int64(0)

In [7]:
df.isnull().sum()

County            0
ClusterNo         0
HHNo              0
interview__key    0
interview__id     0
                 ..
Saving_groups     0
A1                0
has_tele          0
gave_tele         0
hasagr            0
Length: 2353, dtype: int64

## 2. Target Variable Engineering

In [8]:
# we now search the metadata for risk indicators,bu searching for keywords in the column labels
keywords=['loan','default','late','miss','repay','refuse','borrow']
#k means key:column name in our dataset
#v means value:label or description of the column
risk_related={k:v for k,v in meta.column_names_to_labels.items()
              if any (word in v.lower()for word in keywords)}
#we display our findings
for col,label in risk_related.items():
    print(f"{col}:{label}|")


B1B4:B1B4. Loan Security|
B1C3:B1C3. Frequency miss an important family event|
B1EF99:B1EF99. Refused to Answer(DO NOT READ OUT)|
B3D__99:B3D__99  Refused to Answer (DO NOT READ OUT)|
C1_2:C1-2. Savings through mobile banking (e.g. Mshwari , KCB MPesa, MCoop cash, Eazzy Loan, Timiza, HF Whizz)?|
C1_10:C1_10. Registered on Mobile bank (e.g. Mshwari , KCB MPesa, MCoop cash, Eazzy Loan, Timiza, HF Whizz)|
C1_11:C1_11. Personal loan/business loan from a bank|
C1_12:C1_12. Loan from mobile banking (e.g. Mshwari , KCB MPesa, MCoop cash, Eazzy Loan, Timiza, HF Whizz, Stawi loan, M-fanisi)|
C1_13:C1_13. Loan from mobile money provider (e.g Fuliza loan)|
C1_14:C1_14. Loan at a Sacco / Savings and Credit Cooperative organisation|
C1_15:C1_15. Loan from a microfinance institution|
C1_16:C1_16. Loan from Shylocks / Loan Sharks / Money Lenders / Money Merchants that are not from your phone (e.g. Platinum, Ngao, etc.)|
C1_17:C1_17. Loan from a group/chama|
C1_18:C1_18. Loan from a government institu

In [9]:
# We take the metadata and search specifically for performance 'outcomes'
outcome_keywords = ['default', 'late', 'miss', 'distress', 'seized']
target_candidates = {k: v for k, v in risk_related.items() 
                     if any(word in v.lower() for word in outcome_keywords)}

# Print what we found
for col, label in target_candidates.items():
    print(f"{col}: {label}")

B1C3: B1C3. Frequency miss an important family event
E3__1: E3__1  Lack of collateral
E3__18: E3__18  Members defaulting/not cleared loans
E1ivA: E1ivA.Collateral / security for Personal loan from a bank/ business
E1ivC: E1ivB. Collateral / security for Sacco
E1ivD: E1ivD.Collateral / security for -  Microfinance institution loan
E1ivE: E1ivE. Collateral / security for -  Moneylender/Shylock loan
E1ivF: E1ivF.Collateral / security for -   Group/chama loan
E1ivH: E1ivH.Collateral / security for -   Loan from Employer
E1ivO: E1ivO.Collateral / security for -   Loan from Insurance
E1ivOi: E1ivOi.Other collateral / security for   Loan from Insurance
E1iP: E1iP Government or government related institution to buy a house or land
E1iiP: E1iiP. Number of outstanding loans -  Govt or governmentrelated institution
E1iiiP: E1iiiP. MAIN reason for taking loan - Government or govt related ins
E1iiiPi: E1iiiPi. Other Government or govt related institution to buy a house or land
E1ivP: E1ivP.Collater

### 2.1 The Logic of "High Risk"
In credit risk modeling, we must distinguish between **Input Features** (the borrower's characteristics) and **Output Labels** (the borrower's performance). To define our target variable ($y$), we performed a secondary search on the FinAccess 2021 metadata to isolate columns that represent **Credit Failure**.

We filtered the 100+ loan-related results by looking for "Outcome" keywords: *Default*, *Late*, *Missed*, and *Distress*.

### 2.2 Selecting the "Golden Trio"
From the search results, we settled on three primary columns to define a **"High Risk"** individual:

1. **`defaulted` (Defaulted on Loans/Debt):** This represents a "Hard Default" where the contractual obligation was completely broken.
2. **`paidlate` (Paid Late):** This represents "Arrears." In banking, this is an early warning sign of deteriorating creditworthiness.
3. **`E2Ci23` (Paid late/ Missed a payment):** This serves as a "Safety Net" variable to capture any instances of delinquency not recorded in the first two columns.

> **Note:** These three indicators are combined using a logical `OR` operation to create a binary label: 
> * **1 (High Risk):** If the individual meets *any* of the criteria above.
> * **0 (Low Risk):** If the individual has a clean repayment record.

### 2.3 Constructing the Composite Target
Because survey respondents may answer "Yes" to one indicator but "No" to another, we created a **Composite Binary Target** ($y$).

* **Target = 1 (High Risk):** Respondent answered "Yes" to *any* of the three indicators (Defaulted, Paid Late, or Distress).
* **Target = 0 (Low Risk):** Respondent has used credit but has no record of default or late payment.

This approach ensures the model has a robust **"Ground Truth"** to learn the patterns of financial distress in the Kenyan market.

In [10]:
print(f"column E2Ci23 description:{meta.column_names_to_labels.get('E2Ci23')} ")

column E2Ci23 description:Paid late/ Missed a payment 


In [11]:
# we define the risk columns, that we chose earlier
target_columns=['defaulted','paidlate','E2Ci23']

In [12]:
# Now cause survey data mixes strings (yes) and numbers (1), we define what counts as risk value
risk_indicators=['yes',1,1.0,'defaulted']

In [13]:
# we create our composite target
df['is_high_risk'] = df[target_columns].apply(
    lambda row: 1 if any(val in risk_indicators for val in row.values) else 0, 
    axis=1
)

In [14]:
#we now compare the Low and high risk
counts=df['is_high_risk'].value_counts()
percentage=df['is_high_risk'].value_counts(normalize=True)*100
print("--- Target variable distribution ---")
print(f"Low risk(0):{counts[0]}({percentage[0]:.2f}%)")
print(f"High risk(1):{counts[1]}({percentage[1]:.2f}%)")

--- Target variable distribution ---
Low risk(0):14613(66.35%)
High risk(1):7411(33.65%)


### 2.4 Target Distribution and Market Analysis

Upon applying the composite logic, the dataset yielded the following distribution:

| Class | Label | Observations | Percentage |
| :--- | :--- | :--- | :--- |
| **Class 0** | Low Risk | 14,613 | 66.35% |
| **Class 1** | High Risk | 7,411 | 33.65% |



#### Key Observations:

* **High Signal Density:** A risk rate of **~34%** is significantly higher than traditional commercial bank NPL (Non-Performing Loan) ratios, which typically hover below 15%. This suggests that the FinAccess 2021 dataset captures a wide spectrum of the "informal" and "digital" credit market, where risk is more prevalent.
* **Model Feasibility:** From a Machine Learning perspective, this is a **well-balanced dataset**. We do not face a "Needle in a Haystack" problem (common in fraud detection). The model has a substantial number of "High Risk" examples (7,411) to learn the underlying patterns of default.
* **Reflecting the Kenyan Context:** This distribution aligns with the rapid growth of digital lending in Kenya (Fuliza, M-Shwari, and App-based loans), where accessibility is high but repayment distress is a documented socioeconomic challenge.

## 3. Predictor variables

In [15]:
#we now serach the metadata for potential predictor variables
feature_keywords=['age','gender','smartphone','mpesa','educ','income','chama','employmet']
potential_features={k:v for k,v in meta.column_names_to_labels.items()
if any(word in v.lower()for word in feature_keywords)}
for col,label in potential_features.items():
    print(f"{col}:{label}")

A15:A15. Language
A15i:A15i. Language
A17:A17. Not returned to complete their education
A19:A19. AGE OF RESPONDENT
A21:A21. Education Completed
A21i:A21i. Other Education level Completed
A23:A23. Female HH Highest Formal Education
A26__6:A26. Persons with disability:Difficulty in communicating using your usual language
B1EF6:B1EF6. Income from investments (e.g shares,rental)
B3A__1:B3A. Income Sources:farming (crops, keeping livestock, fishing, aquaculture)
B3A__2:B3A. Income Sources:employment
B3A__3:B3A. Income Sources:casual work
B3A__4:B3A. Income Sources:running own business/Self employment
B3A__5:B3A. Income Sources:money from  NGO / Government/Social transfer
B3A__6:B3A. Income Sources:renting,  land, house/rooms, equipment
B3A__7:B3A. Income Sources:Earning money from investments, e.g. shares, stocks
B3A__8:B3A. Income Sources:Pension/Annuity
B3A__9:B3A. Income Sources:Money / support from family / friends / spouse
B3I:B3I. Monthly Income (KSh)
C1_2:C1-2. Savings through mobile

## 3.1 Feature Selection Strategy

To predict the `is_high_risk` target, we selected features across four dimensions:

1. **Demographics:** Age and Education are used as proxies for financial maturity and earning stability.
2. **Economic Indicators:** We look beyond total income to examine *Income Sources* (e.g., Casual work vs. Employment), as cash-flow consistency is a key predictor of repayment.
3. **Digital Behavior:** Given Safaricom's ecosystem, we prioritize mobile money and mobile banking usage metrics as "Alternative Data" for credit scoring.
4. **Social Resilience:** We include participation in informal groups (Chamas) and exposure to recent economic shocks (e.g., job loss or family bereavement) to assess a borrower's support system.

In [16]:
#we now define the features (x) and the variable (y)
selected_columns=[
    #target
    'is_high_risk',

    #demographics
    'A19', #age
    'gender', #gender
    'education', #education level
    #economic
    'B3I', #monthly income
    'incomegpnew',#income group
    'B3A__2', #income souce:employment
    'B3A__3',#income soucre: casual
    #digita/financial behaviour
    'mobile_money_usage',
    'mobile_bank_usage',
    'infgp_usage', #chama usage
    'savings_usage'    
]

In [17]:
#we now slice the dataset to have only the relevant features
df_model=df[selected_columns].copy()

In [18]:
df_model.head(20)

,is_high_risk,A19,gender,education,B3I,incomegpnew,B3A__2,B3A__3,mobile_money_usage,mobile_bank_usage,infgp_usage,savings_usage
0,0,59.0,2.0,4.0,13000.0,5.0,1.0,0.0,1.0,3.0,1.0,1.0
1,0,43.0,2.0,4.0,6000.0,4.0,1.0,0.0,1.0,3.0,1.0,1.0
2,0,72.0,1.0,2.0,1000.0,1.0,0.0,0.0,3.0,3.0,1.0,1.0
3,0,22.0,1.0,2.0,2500.0,2.0,0.0,1.0,1.0,3.0,3.0,1.0
4,1,24.0,1.0,3.0,6000.0,4.0,1.0,0.0,1.0,1.0,1.0,1.0
5,1,50.0,1.0,2.0,4000.0,3.0,0.0,1.0,1.0,3.0,3.0,3.0
6,0,31.0,1.0,2.0,7000.0,4.0,0.0,1.0,1.0,3.0,3.0,1.0
7,0,38.0,1.0,2.0,3000.0,2.0,0.0,0.0,1.0,1.0,3.0,1.0
8,0,31.0,2.0,2.0,3000.0,2.0,0.0,0.0,1.0,3.0,3.0,1.0
9,0,62.0,2.0,1.0,NaN,9.0,0.0,0.0,3.0,3.0,1.0,1.0


In [19]:
#lets get a concise  summary of our new dataset
df_model.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22024 entries, 0 to 22023
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   is_high_risk        22024 non-null  int64  
 1   A19                 22024 non-null  float64
 2   gender              22024 non-null  float64
 3   education           22024 non-null  float64
 4   B3I                 19619 non-null  float64
 5   incomegpnew         22024 non-null  float64
 6   B3A__2              22024 non-null  float64
 7   B3A__3              22024 non-null  float64
 8   mobile_money_usage  22024 non-null  float64
 9   mobile_bank_usage   22024 non-null  float64
 10  infgp_usage         22024 non-null  float64
 11  savings_usage       22024 non-null  float64
dtypes: float64(11), int64(1)
memory usage: 2.0 MB


In [20]:
#lets check for duplicates
df_model.duplicated().sum()

np.int64(2548)

### ____ we see that we have 2548 duplicates,
###  Deduplication Policy
We applied a strict deduplication policy, removing all identical observation rows. This ensures that the model evaluation reflects its ability to **generalize** to new, unseen individuals rather than simply recalling identical profiles present in the training set.

In [21]:
#we opt to drop them to avoid data leakage
df_model=df_model.drop_duplicates()
df_model.duplicated().sum()

np.int64(0)

In [22]:
#lets check for null values
df_model.isnull().sum()

is_high_risk             0
A19                      0
gender                   0
education                0
B3I                   1843
incomegpnew              0
B3A__2                   0
B3A__3                   0
mobile_money_usage       0
mobile_bank_usage        0
infgp_usage              0
savings_usage            0
dtype: int64

### ___ we see that B3I has 2405 null values
### 6.3 Strategic Handling of Missing Values
Rather than performing a simple listwise deletion (dropping 10% of the sample), we opted for a hybrid imputation strategy:
- **Bias Mitigation:** Dropping rows would disproportionately remove informal-sector respondents, leading to selection bias.
- **Numerical Imputation:** Missing Income values were filled with the **Median** to preserve the distribution.

In [23]:
# we opt to fill\impute the missing values using the median
df_model['B3I']=df_model['B3I'].fillna(df_model['B3I'].median())
df_model.isnull().sum()

is_high_risk          0
A19                   0
gender                0
education             0
B3I                   0
incomegpnew           0
B3A__2                0
B3A__3                0
mobile_money_usage    0
mobile_bank_usage     0
infgp_usage           0
savings_usage         0
dtype: int64

In [24]:
# noew that our data is already encoded, we finalize the feature list
features=['A19','gender','education','B3I','incomegpnew','B3A__2','B3A__3', 'mobile_money_usage','mobile_bank_usage','infgp_usage','savings_usage']

In [25]:
#we convert codes to category types to ensure that the model does not see 2.0 for gender as the double of 1.0
categorical_cols=['gender','education','incomegpnew','B3A__2','B3A__3', 'mobile_money_usage','mobile_bank_usage','infgp_usage','savings_usage']
for col in categorical_cols:
    df_model[col]=df_model[col].astype('category')

In [26]:
#we create dummy variables(one hot encoding),this turns 'geneder into 1.0 and 2.0
X=pd.get_dummies(df_model[features],columns=categorical_cols,drop_first=True)
y=df_model['is_high_risk']

In [27]:
X.head()

,A19,B3I,gender_2.0,education_2.0,education_3.0,education_4.0,education_5.0,incomegpnew_2.0,incomegpnew_3.0,incomegpnew_4.0,...,B3A__2_1.0,B3A__3_1.0,mobile_money_usage_2.0,mobile_money_usage_3.0,mobile_bank_usage_2.0,mobile_bank_usage_3.0,infgp_usage_2.0,infgp_usage_3.0,savings_usage_2.0,savings_usage_3.0
0,59.0,13000.0,True,False,False,True,False,False,False,False,...,True,False,False,False,False,True,False,False,False,False
1,43.0,6000.0,True,False,False,True,False,False,False,True,...,True,False,False,False,False,True,False,False,False,False
2,72.0,1000.0,False,True,False,False,False,False,False,False,...,False,False,False,True,False,True,False,False,False,False
3,22.0,2500.0,False,True,False,False,False,True,False,False,...,False,True,False,False,False,True,False,True,False,False
4,24.0,6000.0,False,False,True,False,False,False,False,True,...,True,False,False,False,False,False,False,False,False,False


## Feature Engineering: Categorical Encoding
Since the dataset uses numerical mapping for categories (e.g., Education levels 1-4), we applied **One-Hot Encoding**. 
- **Rationale:** This prevents the model from assuming an "ordinal" relationship where none exists (e.g., assuming Gender 2 is "greater than" Gender 1). 
- **Dimensionality:** After encoding, our feature space expanded to accommodate the unique categories, providing the model with binary inputs for each attribute.

In [28]:
X.columns

Index(['A19', 'B3I', 'gender_2.0', 'education_2.0', 'education_3.0',
       'education_4.0', 'education_5.0', 'incomegpnew_2.0', 'incomegpnew_3.0',
       'incomegpnew_4.0', 'incomegpnew_5.0', 'incomegpnew_6.0',
       'incomegpnew_7.0', 'incomegpnew_8.0', 'incomegpnew_9.0',
       'incomegpnew_10.0', 'B3A__2_1.0', 'B3A__3_1.0',
       'mobile_money_usage_2.0', 'mobile_money_usage_3.0',
       'mobile_bank_usage_2.0', 'mobile_bank_usage_3.0', 'infgp_usage_2.0',
       'infgp_usage_3.0', 'savings_usage_2.0', 'savings_usage_3.0'],
      dtype='object')

In [29]:
X=X.astype(float)
X.head()

,A19,B3I,gender_2.0,education_2.0,education_3.0,education_4.0,education_5.0,incomegpnew_2.0,incomegpnew_3.0,incomegpnew_4.0,...,B3A__2_1.0,B3A__3_1.0,mobile_money_usage_2.0,mobile_money_usage_3.0,mobile_bank_usage_2.0,mobile_bank_usage_3.0,infgp_usage_2.0,infgp_usage_3.0,savings_usage_2.0,savings_usage_3.0
0,59.0,13000.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,43.0,6000.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,72.0,1000.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
3,22.0,2500.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
4,24.0,6000.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
